In [30]:
##### Compiles csv of training data for regions in dataset and does final cleaning

import os
import pandas as pd
from pathlib import Path
import numpy as np
import geopandas as gpd
from functools import reduce

In [31]:
### Set-up

# Get the current working directory
cd = Path.cwd().parent 

# folder of predictors 
predictors = Path(f'{cd}/Data/Clean/Predictors/Vectors')

# intensities 
capital = pd.read_csv(f"{cd}/Data/Clean/Intensities/subnational_capital_intensity.csv")
labor = pd.read_csv(f"{cd}/Data/Clean/Intensities/subnational_labor_intensity.csv")

# save path
save_path_capital = f'{cd}/Data/Clean/Training_data/capital.csv'
save_path_labor = f'{cd}/Data/Clean/Training_data/labor.csv'

In [32]:
##### Merge all predictors into one df (dropping all duplicate PROJ_ID entries first)
dfs = []
for f in os.listdir(predictors):
    if f.endswith(".csv"):
        df = pd.read_csv(os.path.join(predictors, f))
        dupes = df['PROJ_ID'].duplicated().sum()
        if dupes > 0:
            df = df.drop_duplicates(subset='PROJ_ID')
        dfs.append(df)

predictors_df = reduce(lambda left, right: pd.merge(left, right, on='PROJ_ID', how='outer'), dfs)

In [33]:
### Merge with capital and labor 

### Capital
capital = capital.drop_duplicates(subset='PROJ_ID')
capital_merge = capital.merge(predictors_df, on='PROJ_ID', how='left')

### Labor
labor = labor.drop_duplicates(subset='PROJ_ID')
labor_merge = labor.merge(predictors_df, on='PROJ_ID', how='left')


In [34]:
#### Drop compositional variables

# one of each of field size %'s and production mix %'s 
# share_vlarge_field, other_share_production_USD  

# also drop aggregate variables

col_to_drop = ['GEO_ID_NAME', 'total_production_USD', 'share_vlarge_field', 'other_share_production_USD']

capital_merge = capital_merge.drop(columns = col_to_drop)
capital_merge = capital_merge.drop(columns = ['ag_capital_stock_USD_nominal'])

labor_merge = labor_merge.drop(columns = col_to_drop)
labor_merge = labor_merge.drop(columns = ['ag_jobs'])


### Drop regions missing data
capital_merge = capital_merge.dropna()
labor_merge = labor_merge.dropna()

### Add country_ID columns 
capital_merge['country_ID'] = capital_merge['PROJ_ID'].str[:3]
labor_merge['country_ID'] = labor_merge['PROJ_ID'].str[:3]

### Convert intensities to log
capital_merge["log_capital_intensity_USD_per_USD"] = np.log(capital_merge["capital_intensity_USD_per_USD"])
capital_merge = capital_merge.drop(columns=["capital_intensity_USD_per_USD"])

labor_merge["log_labor_intensity_jobs_per_million_USD"] = np.log(labor_merge["labor_intensity_jobs_per_million_USD"])
labor_merge = labor_merge.drop(columns=["labor_intensity_jobs_per_million_USD"])

In [35]:
### Save
capital_merge.to_csv(save_path_capital, index=False)
labor_merge.to_csv(save_path_labor, index=False)